In [16]:
import datetime
import json

import ephem
import numpy as np
import pandas as pd
import pytz

In [17]:
EPHEM_EPOCH = datetime.datetime(1899, 12, 31)


def _as_float(value):
    """Convert ephem angle/date-ish values to plain Python floats."""
    return float(value)


def rad_to_deg(rad):
    return float(np.degrees(_as_float(rad)))


def rad_to_hours(rad):
    return float(_as_float(rad) * 12 / np.pi)


def au_to_km(au):
    return float(_as_float(au) * 149597870.7)


def ephem_to_iso(ephem_date):
    dt = EPHEM_EPOCH + datetime.timedelta(days=_as_float(ephem_date))
    return dt.isoformat()


def get_moon_raw_parameters(obs_date, lat, lon, elevation=0, pressure=1013.25, horizon='-0:34', epoch='2000'):
    """Compute raw Moon parameters from PyEphem for a given observer."""
    observer = ephem.Observer()
    observer.lat = str(lat)
    observer.lon = str(lon)
    observer.elev = elevation
    observer.pressure = pressure
    observer.horizon = horizon
    observer.epoch = epoch
    observer.date = obs_date

    moon = ephem.Moon()
    moon.compute(observer)

    raw = {}
    for attr in dir(moon):
        if attr.startswith("_"):
            continue
        try:
            value = getattr(moon, attr)
            if callable(value):
                continue
            raw[attr] = value
        except Exception:
            continue
    return raw


def format_moon_output(raw):
    """Normalize raw ephemeris output for API/frontend usage."""
    altitude_rad = _as_float(raw["alt"])

    return {
        "meta": {
            "object": str(raw["name"]),
        },
        "position": {
            "equatorial": {
                "ra_hours": rad_to_hours(raw["ra"]),
                "dec_deg": rad_to_deg(raw["dec"]),
                "hour_angle_hours": rad_to_hours(raw["ha"]),
            },
            "geocentric": {
                "ra_hours": rad_to_hours(raw["g_ra"]),
                "dec_deg": rad_to_deg(raw["g_dec"]),
            },
            "horizontal": {
                "altitude_deg": rad_to_deg(raw["alt"]),
                "azimuth_deg": rad_to_deg(raw["az"]),
            },
        },
        "phase": {
            "illumination_fraction": _as_float(raw["moon_phase"]),
            "phase_percent": _as_float(raw["phase"]),
            "elongation_deg": rad_to_deg(raw["elong"]),
            "phase_angle_deg": 180.0 - rad_to_deg(raw["elong"]),
            "visual_magnitude": _as_float(raw["mag"]),
        },
        "distance": {
            "au": _as_float(raw["earth_distance"]),
            "km": au_to_km(raw["earth_distance"]),
        },
        "libration": {
            "latitude_deg": rad_to_deg(raw["libration_lat"]),
            "longitude_deg": rad_to_deg(raw["libration_long"]),
            "terminator_tilt_deg": rad_to_deg(raw["libration_long"]),
        },
        "appearance": {
            "angular_radius_deg": rad_to_deg(raw["radius"]),
            "angular_size_arcsec": _as_float(raw["size"]),
            "colongitude_deg": rad_to_deg(raw["colong"]),
            "subsolar_lat_deg": rad_to_deg(raw["subsolar_lat"]),
        },
        "visibility": {
            "rise_time": ephem_to_iso(raw["rise_time"]),
            "set_time": ephem_to_iso(raw["set_time"]),
            "transit_time": ephem_to_iso(raw["transit_time"]),
            "rise_azimuth_deg": rad_to_deg(raw["rise_az"]),
            "set_azimuth_deg": rad_to_deg(raw["set_az"]),
            "transit_altitude_deg": rad_to_deg(raw["transit_alt"]),
            "visibility_score": max(0.0, float(np.sin(altitude_rad))),
            "circumpolar": bool(raw["circumpolar"]),
            "never_up": bool(raw["neverup"]),
        },
        "solar_geometry": {
            "sun_distance_au": _as_float(raw["sun_distance"]),
        },
    }


def get_moon_parameters_dataframe(raw):
    data = [{"attribute": key, "value": value} for key, value in raw.items()]
    return pd.DataFrame(data)
    

In [19]:
tz = pytz.timezone("Asia/Karachi")
local_date = datetime.datetime.now(tz)

raw_moon = get_moon_raw_parameters(local_date, 24.86, 66.99)
moon_table = get_moon_parameters_dataframe(raw_moon)
formatted_moon = format_moon_output(raw_moon)

print(moon_table)
print("\nStructured Moon Output:\n")
print(json.dumps(formatted_moon, indent=2))

         attribute         value
0            a_dec     -0.111033
1          a_epoch       36524.5
2             a_ra      3.277873
3              alt     -1.115681
4               az      5.419963
5      circumpolar         False
6           colong      0.980319
7              dec     -0.118648
8   earth_distance      0.002679
9            elong      2.636308
10           g_dec     -0.113582
11            g_ra       3.28382
12              ha      2.798491
13            hlat     -0.047986
14            hlon       3.31696
15           hlong       3.31696
16   libration_lat      0.064033
17  libration_long      0.083846
18             mag        -11.87
19      moon_phase      0.937453
20            name          Moon
21         neverup         False
22           phase     93.780716
23              ra      3.278857
24          radius      0.004342
25         rise_az      1.728816
26       rise_time   46140.00232
27          set_az      4.501939
28        set_time  46140.493199
29        